In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
import warnings
warnings.filterwarnings("ignore")
import os

In [12]:
os.makedirs("plots/macro", exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=1.1)

In [13]:
df = pd.read_csv("tshirts_preprocessed.csv", parse_dates=["week_start"])

In [14]:
results = []
def log(msg):
    print(msg)
    results.append(msg)
 
log("=" * 65)
log("MACROECONOMIC INFLUENCE ANALYSIS")
log("=" * 65)
 
MACRO_COLS = ["eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci"]
MACRO_LABELS = {
    "eurozone_hicp":               "HICP Inflation (%)",
    "eurozone_unemployment_rate":  "Unemployment Rate (%)",
    "eurozone_cci":                "Consumer Confidence Index",
}

MACROECONOMIC INFLUENCE ANALYSIS


In [15]:
monthly = (
    df.groupby("year_month")
    .agg(
        total_sales=("weekly_sales_volume", "sum"),
        mean_sales=("weekly_sales_volume", "mean"),
        **{col: (col, "mean") for col in MACRO_COLS}
    )
    .reset_index()
    .sort_values("year_month")
)
monthly["date"] = pd.to_datetime(monthly["year_month"])
log(f"\n  Monthly observations: {len(monthly)}")


  Monthly observations: 25


In [17]:
log("\n" + "─" * 50)
log("A. Pearson Correlations — Macro Indicators vs. Monthly Sales")
log("─" * 50)
 
corr_df = monthly[MACRO_COLS + ["total_sales", "mean_sales"]].copy()
corr_matrix = corr_df.corr()
 
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
    center=0, vmin=-1, vmax=1,
    xticklabels=[MACRO_LABELS.get(c, c) for c in corr_matrix.columns],
    yticklabels=[MACRO_LABELS.get(c, c) for c in corr_matrix.index],
    linewidths=0.5, ax=ax
)
ax.set_title("Correlation Matrix: Macro Indicators × Sales",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/macro/A1_correlation_matrix.png")
plt.close()
for col in MACRO_COLS:
    r, p = stats.pearsonr(monthly[col], monthly["total_sales"])
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    log(f"  {MACRO_LABELS[col]:<30}  r = {r:+.3f}   p = {p:.4f}  {sig}")
log("\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant")
print("Saved: A1_correlation_matrix.png")


──────────────────────────────────────────────────
A. Pearson Correlations — Macro Indicators vs. Monthly Sales
──────────────────────────────────────────────────
  HICP Inflation (%)              r = +0.269   p = 0.1939  n.s.
  Unemployment Rate (%)           r = -0.054   p = 0.7995  n.s.
  Consumer Confidence Index       r = -0.100   p = 0.6360  n.s.

  Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant
Saved: A1_correlation_matrix.png


In [18]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
 
for ax, col in zip(axes, MACRO_COLS):
    x = monthly[col]
    y = monthly["total_sales"]
    ax.scatter(x, y, alpha=0.7, color="steelblue", s=60, edgecolors="white")
 
    # Regression line
    m, b, r, p, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color="tomato", linewidth=2,
            label=f"r={r:.2f}, p={p:.3f}")
 
    ax.set_xlabel(MACRO_LABELS[col])
    ax.set_ylabel("Monthly Total Sales")
    ax.set_title(MACRO_LABELS[col], fontweight="bold")
    ax.legend(fontsize=9)
 
plt.suptitle("T-Shirt Monthly Sales vs. Macroeconomic Indicators",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/macro/B1_scatter_macro_vs_sales.png")
plt.close()
print("Saved: B1_scatter_macro_vs_sales.png")

Saved: B1_scatter_macro_vs_sales.png


In [19]:
log("\n" + "─" * 50)
log("C. Granger Causality Tests (macro → sales, lags 1–4 months)")
log("─" * 50)
 
def adf_test(series, name):
    result = adfuller(series.dropna())
    stat, p = result[0], result[1]
    log(f"  ADF stationarity — {name}: stat={stat:.3f}  p={p:.4f}  "
        f"{'STATIONARY' if p < 0.05 else 'NON-STATIONARY (consider differencing)'}")
    return p < 0.05
 
log("\n  Stationarity checks (required for Granger test):")
sales_stationary = adf_test(monthly["total_sales"], "total_sales")
for col in MACRO_COLS:
    adf_test(monthly[col], MACRO_LABELS[col])
 
# Difference all series for Granger test
# HICP is stationary but differencing a stationary series keeps it stationary;
# unemployment and CCI are non-stationary and must be differenced.
monthly_d = monthly.copy()
monthly_d["total_sales_d"] = monthly_d["total_sales"].diff()
for col in MACRO_COLS:
    monthly_d[col + "_d"] = monthly_d[col].diff()
monthly_d = monthly_d.dropna()
 
log("\n  Note: non-stationary series (Unemployment, CCI) are first-differenced before Granger testing.")
log(f"  Statistical power caveat: with n={len(monthly_d)} observations after differencing and up to {MAX_LAG} lags, "
    "power is limited — results are indicative rather than conclusive.")
log("\n  Granger causality results (H0: macro does NOT cause sales):")
MAX_LAG = 4
for col in MACRO_COLS:
    data = monthly_d[[col + "_d", "total_sales_d"]].dropna()
    try:
        gc_result = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
        log(f"\n  {MACRO_LABELS[col]}:")
        for lag in range(1, MAX_LAG + 1):
            f_stat = gc_result[lag][0]["ssr_ftest"][0]
            p_val  = gc_result[lag][0]["ssr_ftest"][1]
            sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "n.s."))
            log(f"    Lag {lag}: F={f_stat:.3f}  p={p_val:.4f}  {sig}")
    except Exception as e:
        log(f"  {col}: could not run Granger test — {e}")

In [20]:
log("\n" + "─" * 50)
log("D. Cross-Correlation — Macro leads Sales by N months")
log("─" * 50)
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lags_range = range(-3, 5)  # negative = sales leads macro, positive = macro leads sales
 
for ax, col in zip(axes, MACRO_COLS):
    cc_values = []
    for lag in lags_range:
        shifted = monthly[col].shift(lag)
        valid = monthly["total_sales"].align(shifted, join="inner")
        mask = shifted.notna()
        if mask.sum() > 5:
            r, _ = stats.pearsonr(monthly["total_sales"][mask], shifted[mask])
        else:
            r = np.nan
        cc_values.append(r)
 
    colors = ["tomato" if abs(v) == max(abs(c) for c in cc_values if not np.isnan(c)) else "steelblue"
              for v in cc_values]
    ax.bar(list(lags_range), cc_values, color=colors, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(MACRO_LABELS[col], fontweight="bold")
    ax.set_xlabel("Lag (months, positive = macro leads sales)")
    ax.set_ylabel("Pearson r")
 
    best_lag = list(lags_range)[np.nanargmax([abs(v) for v in cc_values])]
    log(f"  {MACRO_LABELS[col]}: strongest correlation at lag {best_lag} months  (r = {cc_values[best_lag - min(lags_range)]:.3f})")
 
plt.suptitle("Cross-Correlation: Macro Indicators → T-Shirt Sales\n(positive lag = macro precedes sales)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/macro/D1_cross_correlation.png")
plt.close()
print("Saved: D1_cross_correlation.png")


──────────────────────────────────────────────────
D. Cross-Correlation — Macro leads Sales by N months
──────────────────────────────────────────────────
  HICP Inflation (%): strongest correlation at lag 2 months  (r = 0.389)
  Unemployment Rate (%): strongest correlation at lag 4 months  (r = 0.276)
  Consumer Confidence Index: strongest correlation at lag -1 months  (r = -0.175)
Saved: D1_cross_correlation.png


In [21]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
 
# Top: total weekly sales
weekly_agg = df.groupby("week_start")["weekly_sales_volume"].sum().reset_index()
axes[0].fill_between(weekly_agg["week_start"], weekly_agg["weekly_sales_volume"],
                     alpha=0.3, color="steelblue")
axes[0].plot(weekly_agg["week_start"], weekly_agg["weekly_sales_volume"],
             color="steelblue", linewidth=1.5)
axes[0].set_ylabel("Weekly Sales Volume")
axes[0].set_title("T-Shirt Sales & Consumer Confidence — COVID Shock Period",
                  fontweight="bold")
 
# Bottom: CCI over time
cci_weekly = df.groupby("week_start")["eurozone_cci"].mean().reset_index()
axes[1].plot(cci_weekly["week_start"], cci_weekly["eurozone_cci"],
             color="darkorange", linewidth=2)
axes[1].fill_between(cci_weekly["week_start"], cci_weekly["eurozone_cci"],
                     alpha=0.2, color="darkorange")
axes[1].set_ylabel("Consumer Confidence Index")
axes[1].set_xlabel("Week")
 
for ax in axes:
    ax.axvline(pd.Timestamp("2020-03-01"), color="red", linestyle="--",
               linewidth=1.5, label="COVID-19 shock")
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
 
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("plots/macro/E1_covid_shock.png")
plt.close()
print("Saved: E1_covid_shock.png")

Saved: E1_covid_shock.png


In [22]:
with open("macro_analysis_results.txt", "w") as f:
    f.write("\n".join(results))
print("\nMacro analysis complete. Results saved to macro_analysis_results.txt")


Macro analysis complete. Results saved to macro_analysis_results.txt
